In [12]:
!pip install ragas[all]
!pip install litellm

In [13]:
# Import libraries and setup config
import os
import pandas as pd
from google.colab import userdata
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecision, ContextRecall, Faithfulness, AnswerCorrectness, FactualCorrectness
from ragas.embeddings.base import embedding_factory

# Set the OPENAI_API_KEY environment variable from Colab secrets
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Setup LLM
client = AsyncOpenAI()
# Increase max_tokens for the LLM to prevent truncation of structured outputs
llm = llm_factory("gpt-4o-mini", client=client, max_tokens=4096)
embeddings = embedding_factory("openai", model="text-embedding-3-small", client=client)


In [14]:
# Import data and pre-process

qa_data_raw = pd.read_csv('/content/drive/MyDrive/RAG/datasets/rag_with_deepseek_result_20251218_1016.csv')
qa_data_raw = qa_data_raw[['query', 'answer_ref', 'answer', 'content', 'passage_1_text']]
qa_data_grouped = qa_data_raw.groupby(['query', 'answer_ref', 'answer', 'passage_1_text'])['content'].agg(list).reset_index()
qa_data = qa_data_grouped.rename(columns={'query': 'user_input',
                                                  'passage_1_text': 'reference',
                                                  'content': 'retrieved_contexts'})
# display(qa_data.head())

qa_data_dict = qa_data.to_dict()

qa_data_expanded = qa_data['retrieved_contexts'].apply(lambda x: pd.Series(x + [None]*(4-len(x)) if len(x) < 4 else x[:4]))
qa_data_expanded.columns = [f'retrieved_contexts_{i+1}' for i in range(4)]
qa_data_final = pd.concat([qa_data.drop(columns=['retrieved_contexts']), qa_data_expanded], axis=1)

display(qa_data_final.head())





,user_input,answer_ref,answer,reference,retrieved_contexts_1,retrieved_contexts_2,retrieved_contexts_3,retrieved_contexts_4
0,"""According to the passages, during what time p...","""The second school closure occurred from Decem...",I need to answer based ONLY on the provided co...,"""Then primary/elementary schools were closed o...",FRANCE \nPIRLS 2021 ENCYCLOPEDIA 13 \n \n \nTh...,FRANCE \nPIRLS 2021 ENCYCLOPEDIA 13 \n \n \nTh...,FRANCE \nPIRLS 2021 ENCYCLOPEDIA 13 \n \n \nTh...,FRANCE \nPIRLS 2021 ENCYCLOPEDIA 13 \n \n \nTh...
1,"""According to the passages, how did the COVID-...","""During the COVID-19 pandemic, many educationa...","According to the passages, the COVID-19 pandem...","""Due to disruptions caused by the COVID-19 pan...","student achievement, including NAEP, were post...","student achievement, including NAEP, were post...","student achievement, including NAEP, were post...","student achievement, including NAEP, were post..."
2,"""According to the passages, what constitutiona...","""Freedom of education (guaranteed by the Dutch...","According to the passages, the constitutional ...","""One of the key features of the Dutch educatio...",NETHERLANDS \nPIRLS 2021 ENCYCLOPEDIA 1 \n \n ...,NETHERLANDS \nPIRLS 2021 ENCYCLOPEDIA 1 \n \n ...,NETHERLANDS \nPIRLS 2021 ENCYCLOPEDIA 1 \n \n ...,NETHERLANDS \nPIRLS 2021 ENCYCLOPEDIA 1 \n \n ...
3,"""According to the passages, what is the name o...","""The law is the Every Student Succeeds Act (ES...","ESSA, the Every Student Succeeds Act, is the l...","""In December 2015, Congress reauthorized ESEA ...","regulations, elected or appointed school board...","regulations, elected or appointed school board...","regulations, elected or appointed school board...","regulations, elected or appointed school board..."
4,"""According to the passages, what specific type...","""The SkillsFuture Work-Study Programs, which p...",The specific type of pathway mentioned is the ...,"""In addition to providing preemployment traini...",providers of continuing education and training...,providers of continuing education and training...,providers of continuing education and training...,providers of continuing education and training...


## Evaluation

### Context Precision and Recall

In [ ]:
%%time
import asyncio
import nest_asyncio

context_precision_scorer = ContextPrecision(llm=llm)
context_recall_scorer = ContextRecall(llm=llm)

context_precision_scores = []
context_recall_scores = []

async def calculate_scores():
    for index, row in qa_data.iterrows():
        context_precision_result = await context_precision_scorer.ascore(
            user_input=row['user_input'],
            reference=row['reference'],
            retrieved_contexts=row['retrieved_contexts'],
        )

        context_precision_scores.append(context_precision_result.value)

        context_recall_result = await context_recall_scorer.ascore(
            user_input=row['user_input'],
            reference=row['reference'],
            retrieved_contexts=row['retrieved_contexts'],
        )

        context_recall_scores.append(context_recall_result.value)

nest_asyncio.apply()
asyncio.run(calculate_scores())

qa_data['context_precision_score'] = context_precision_scores
qa_data['context_recall_score'] = context_recall_scores

CPU times: user 2min 18s, sys: 2.53 s, total: 2min 20s
Wall time: 48min 46s


In [ ]:
# Save result to CSV
qa_data.to_csv('/content/drive/MyDrive/RAG/results/eval_precision_recall_deepseek_20251220.csv', index=False)

### Faithfulness

In [ ]:
%%time
import asyncio
import nest_asyncio

faithfulness_scorer = Faithfulness(llm=llm)
faithfulness_scores = []

async def calculate_faithfulness():
    for index, row in qa_data.iterrows():
        faithfulness_result = await faithfulness_scorer.ascore(
            user_input=row['user_input'],
            response=row['answer'],
            retrieved_contexts=row['retrieved_contexts'],
        )

        faithfulness_scores.append(faithfulness_result.value)

nest_asyncio.apply()
asyncio.run(calculate_faithfulness())

qa_data['faithfulness_score'] = faithfulness_scores



CPU times: user 49.6 s, sys: 795 ms, total: 50.4 s
Wall time: 55min 9s


In [ ]:
qa_data.columns

Index(['user_input', 'answer_ref', 'answer', 'reference', 'retrieved_contexts',
       'faithfulness_score'],
      dtype='object')

In [ ]:
# Save result to CSV
qa_data.to_csv('/content/drive/MyDrive/RAG/results/eval_with_faithfulness_deepseek_20251220.csv', index=False)


### Answer Correctness

In [ ]:
%%time
import asyncio
import nest_asyncio

answer_correctness_scorer = AnswerCorrectness(llm=llm, embeddings=embeddings)
answer_correctness_scores = []

async def calculate_answer_correctness():
    for index, row in qa_data.iterrows():
        answer_correctness_result = await answer_correctness_scorer.ascore(
            user_input=row['user_input'],
            response=row['answer'],
            reference=row['reference'],
        )

        answer_correctness_scores.append(answer_correctness_result.value)

nest_asyncio.apply()
asyncio.run(calculate_answer_correctness())

qa_data['answer_correctness_score'] = answer_correctness_scores

CPU times: user 2min 1s, sys: 2.06 s, total: 2min 3s
Wall time: 1h 30min 5s


In [ ]:
# Save result to CSV
qa_data.to_csv('/content/drive/MyDrive/RAG/results/eval_with_answer_correctness_deepseek_20251220.csv', index=False)

### Factual Correctness

In [15]:
%%time
import asyncio
import nest_asyncio

factual_correctness_scorer = FactualCorrectness(llm=llm)
factual_correctness_scores = []

async def calculate_factual_correctness():
    for index, row in qa_data.iterrows():
        factual_correctness_result = await factual_correctness_scorer.ascore(
            response=row['answer'],
            reference=row['reference'],
            )

        factual_correctness_scores.append(factual_correctness_result.value)

nest_asyncio.apply()
asyncio.run(calculate_factual_correctness())

qa_data['factual_correctness_score'] = factual_correctness_scores

CPU times: user 1min 39s, sys: 1.64 s, total: 1min 40s
Wall time: 1h 29min 55s


In [ ]:
# Save result to CSV
qa_data.to_csv('/content/drive/MyDrive/RAG/results/eval_with_factual_correctness_deepseek_20251220.csv', index=False)